In [5]:
from pymongo.collection import Collection
from pymongo import MongoClient
import pandas as pd
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score


model = LogisticRegression()

all_trained_models = []

all_score = []

all_training_index = []
all_testing_index = []

file_to_read = open("features/all_training_features.pickle", "rb")
X_train = pickle.load(file_to_read)
file_to_read.close()

file_to_read = open("features/all_training_labels.pickle", "rb")
y_train = pickle.load(file_to_read)
file_to_read.close()

file_to_read = open("features/all_testing_features.pickle", "rb")
X_test = pickle.load(file_to_read)
file_to_read.close()

file_to_read = open("features/all_testing_labels.pickle", "rb")
y_test = pickle.load(file_to_read)
file_to_read.close()

X_train_df = pd.concat(X_train)
y_train_df = pd.concat(y_train)

X_test_df = pd.concat(X_test)
y_test_df = pd.concat(y_test)

model.fit(X_train_df, y_train_df.values.ravel())

collection: Collection = MongoClient().wesad.centralized

mongo_dict = {
    "individual_scoring": list(),
    "centralized_scoring": dict()
}

for idx in range(15):
    file_to_read = open("features/testing_features"+str(idx)+".pickle", "rb")
    X_test = pickle.load(file_to_read)
    file_to_read.close()

    file_to_read = open("features/testing_labels"+str(idx)+".pickle", "rb")
    y_test = pickle.load(file_to_read)
    file_to_read.close()

    print(idx)
    y_pred = model.predict(X_test)
    acc=accuracy_score(y_test, y_pred)
    prec=precision_score(y_test, y_pred, pos_label=2)
    rec=recall_score(y_test, y_pred, pos_label=2)
    f1=f1_score(y_test, y_pred, pos_label=2)
    print(acc, prec, rec, f1)
    
    mongo_dict["individual_scoring"].append({
        "subject_number": idx,
        "acc": acc,
        "prec": prec,
        "rec": rec,
        "f1": f1
    })

print("centralized results")
y_pred = model.predict(X_test_df)
acc=accuracy_score(y_test_df, y_pred)
prec=precision_score(y_test_df, y_pred, pos_label=2)
rec=recall_score(y_test_df, y_pred, pos_label=2)
f1=f1_score(y_test_df, y_pred, pos_label=2)
print(acc, prec, rec, f1)

mongo_dict["centralized_scoring"] = {
    "acc": acc,
    "prec": prec,
    "rec": rec,
    "f1": f1
}

collection.insert_one(mongo_dict)


/Users/cedric/miniconda3/envs/wesad/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0
0.9736164736164736 0.9381443298969072 0.9763948497854077 0.9568874868559412
1
0.9721695129664769 0.9393305439330544 0.9676724137931034 0.9532908704883227
2
0.9848961611076148 0.9844444444444445 0.9630434782608696 0.9736263736263736
3
0.992467043314501 0.9977578475336323 0.9758771929824561 0.9866962305986696
4
0.9833230389129092 0.951417004048583 0.9936575052854123 0.9720785935884177
5
0.8848297213622291 0.7236641221374046 0.9895615866388309 0.8359788359788359
6
0.9359605911330049 0.9588100686498856 0.8297029702970297 0.8895966029723992
7
0.9337461300309597 0.9971264367816092 0.7660044150110376 0.8664169787765293
8
0.9411411411411411 1.0 0.8274647887323944 0.905587668593449
9
0.9664429530201343 0.8982035928143712 0.9911894273127754 0.9424083769633508
10
0.9938949938949939 0.981203007518797 1.0 0.9905123339658444
11
0.8791946308724832 0.8567774936061381 0.7023060796645703 0.771889400921659
12
0.9543517954960439 0.9385593220338984 0.9059304703476483 0.9219562955254943
13
0.9638922888616

InsertOneResult(ObjectId('6644736216c6a0c0ed86bc30'), acknowledged=True)